# 09 · Inferência — Gravar Previsões na Gold

Usa os modelos/decisões do portão já calculados no notebook `08`
(`%run` — reaproveita tudo, não retreina nada) pra gerar a previsão
**real** (a partir do último dia de dado disponível, não mais avaliação
sobre teste histórico) e grava em `gold.previsoes_incidentes`.

**Regra simples:** se o vencedor daquela combinação foi o **modelo**,
usa o modelo pra prever; se foi a **baseline**, a própria baseline
(`media_movel_7d`) já É a previsão — sem misturar as duas.

A linha usada como ponto de partida (a mais recente disponível, por
segmento) vem do `teste_bruto` retornado pelo `08` — que **não** passou
pelo `dropna` de alvo, então inclui o dia mais recente de verdade (que
não tem `target` preenchido, por construção, mas tem as features de
entrada normalmente). Chamamos só `pipeline_ajustado.transform(...)`
direto nele — o `Pipeline` cuida de indexação e montagem do vetor de
features sozinho, sem precisar de nada pré-processado.

In [ ]:
%run ./00_config

In [ ]:
%run ./08_train_volume

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from datetime import timedelta

## Gerar a previsão real por combinação

Pega, de cada `teste_df`, só a linha da **última data disponível por
segmento** (não necessariamente o mesmo dia pra todo segmento, embora
na prática costume ser — como as tabelas nascem de um cross completo
com o calendário, todo segmento tem linha até a mesma data máxima).

In [ ]:
linhas_previsao = []

for nome_tabela, coluna_segmento in configuracoes:
    for horizonte in horizontes:
        chave = f"{nome_tabela}__{horizonte}"
        resultado = resultados_por_chave[chave]
        teste_bruto = resultado["teste_bruto"]

        colunas_segmentacao_reais = [c for c in [coluna_segmento, "prioridade_num"] if c]
        w_ultima_data = F.max("data_abertura").over(Window.partitionBy(*colunas_segmentacao_reais))
        ultima_por_segmento = (
            teste_bruto.withColumn("_data_max_segmento", w_ultima_data)
            .filter(F.col("data_abertura") == F.col("_data_max_segmento"))
            .drop("_data_max_segmento")
        )

        if resultado["vencedor"] == "modelo":
            previsto = resultado["pipeline_ajustado"].transform(ultima_por_segmento).withColumnRenamed("prediction", "valor_previsto")
        else:
            previsto = ultima_por_segmento.withColumn("valor_previsto", F.col("media_movel_7d"))

        dias_futuro = 1 if horizonte == "target_d1" else 7
        previsto = previsto.withColumn("data_prevista", F.date_add("data_abertura", dias_futuro))

        selecao = [
            F.col("data_prevista"),
            F.lit(horizonte).alias("horizonte"),
            F.lit(nome_tabela).alias("tabela_origem"),
            F.lit(resultado["vencedor"]).alias("metodo"),
            F.round("valor_previsto", 2).alias("valor_previsto"),
            (F.col("produto") if "produto" in previsto.columns else F.lit(None).cast("string")).alias("produto"),
            (F.col("categoria") if "categoria" in previsto.columns else F.lit(None).cast("string")).alias("categoria"),
            F.col("prioridade_num"),
        ]
        linhas_previsao.append(previsto.select(*selecao))
        print(f"{chave}: {resultado['vencedor']} | {previsto.count()} linhas de previsão geradas")

previsoes_finais = linhas_previsao[0]
for df_extra in linhas_previsao[1:]:
    previsoes_finais = previsoes_finais.unionByName(df_extra)

previsoes_finais = previsoes_finais.withColumn("data_geracao", F.current_timestamp())

print(f"\nTotal de linhas de previsão: {previsoes_finais.count()}")

## Gravar em `gold.previsoes_incidentes`

Schema com `produto`/`categoria` opcionais (nulos conforme a origem) —
não é uma dimensão tradicional, é uma tabela de saída de aplicação,
então esse formato "esparso" é aceitável aqui (diferente da Gold
dimensional do notebook `06`, que não deveria ter nulos assim).

In [ ]:
(
    previsoes_finais.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(qualified_table(SCHEMA_GOLD, "previsoes_incidentes"))
)

print(f"gold.previsoes_incidentes gravada: {previsoes_finais.count()} linhas")
display(spark.table(qualified_table(SCHEMA_GOLD, "previsoes_incidentes")).orderBy("tabela_origem", "horizonte"))